Optimizing Model Parameters
===========================

Now that we have a model and data it\'s time to train, validate and test
our model by optimizing its parameters on our data. Training a model is
an iterative process; in each iteration the model makes a guess about
the output, calculates the error in its guess (*loss*), collects the
derivatives of the error with respect to its parameters (as we saw in
the [previous section](autograd_tutorial.html)), and **optimizes** these
parameters using gradient descent. For a more detailed walkthrough of
this process, check out this video on [backpropagation from
3Blue1Brown](https://www.youtube.com/watch?v=tIeHLnjs5U8).




## Exercise 1 - Import all the required libraries

From any of the previous notebooks, we can import the libraries and modules required for this section.


<details><summary><b>Solution</b></summary> <pre>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
``` 
</pre></details>

## Excercise 2 - Load the data

From the previous 08_Data, we can load the data and create the data loaders. 

Create a data loader import the FashinoMNIST dataset and create a data loader for the training and test set. Remembe to use transforms to pass the data to tensors

<details><summary><b>Solution</b></summary> <pre>

```python
# Define a transform to normalize the data
transform = transforms.Compose([transforms.ToTensor()])

# Download and load the training data
trainset = datasets.FashionMNIST(root='../data', 
                                train=True, 
                                download=True, 
                                transform=transform
                                )

train_dataloader = torch.utils.data.DataLoader(trainset, 
                                        batch_size=64, 
                                        shuffle=True
                                        )

# Download and load the test data
testset = datasets.FashionMNIST(root='../data', 
                                train=False, 
                                download=True, 
                                transform=transform
                                )
                                
test_dataloader = torch.utils.data.DataLoader(testset, 
                                         batch_size=64, 
                                         shuffle=False
                                         )
``` 
</pre></details>

## Exercise 3 - Define the model
From the previous 09_BuildModel, we can define the model.

Copy the code to define the model , remember to define the device to be used. Finally move the model to the device.

## Exercise 4 - Define the device to be used

**Define the Device**: Check for the availability of MPS, CUDA, MTIA, and XPU in that order, and set the device accordingly. If none of these are available, default to using the CPU.

<details><summary><b>Solution</b></summary> <pre>

```python

# Check if MPS (Metal Performance Shaders) is available
if torch.backends.mps.is_available():
    device = torch.device("mps")
# Check if CUDA is available
elif torch.cuda.is_available():
    device = torch.device("cuda")
# Check if MTIA is available (Meta Training and Inference Accelerator)
elif torch.mtia.is_available():
    device = torch.device("mtia")
# Check if XPU is available ((Intel's heterogeneous computing platform).)
elif torch.xpu.is_available():
    device = torch.device("xpu")
# Default to CPU if no accelerators are available
else:
    device = torch.device("cpu")

print(f"Using device: {device}")


class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
``` 
</pre></details>

Hyperparameters
===============

Hyperparameters are adjustable parameters that let you control the model
optimization process. Different hyperparameter values can impact model
training and convergence rates ([read
more](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html)
about hyperparameter tuning)

We define the following hyperparameters for training:
-   **Number of Epochs** - the number times to iterate over the
        dataset
-   **Batch Size** - the number of data samples propagated through
        the network before the parameters are updated
-   **Learning Rate** - how much to update models parameters at each batch/epoch. Smaller values yield slow learning speed, while
        large values may result in unpredictable behavior during
        training.

We are going to use the following hyperparameters:

```python
learning_rate = 1e-3
batch_size = 64
epochs = 5
```




Loss Function
-------------

When presented with some training data, our untrained network is likely
not to give the correct answer. **Loss function** measures the degree of
dissimilarity of obtained result to the target value, and it is the loss
function that we want to minimize during training. To calculate the loss
we make a prediction using the inputs of our given data sample and
compare it against the true data label value.

Common loss functions include
[nn.MSELoss](https://pytorch.org/docs/stable/generated/torch.nn.MSELoss.html#torch.nn.MSELoss)
(Mean Square Error) for regression tasks, and
[nn.NLLLoss](https://pytorch.org/docs/stable/generated/torch.nn.NLLLoss.html#torch.nn.NLLLoss)
(Negative Log Likelihood) for classification.
[nn.CrossEntropyLoss](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html#torch.nn.CrossEntropyLoss)
combines `nn.LogSoftmax` and `nn.NLLLoss`.

We pass our model\'s output logits to `nn.CrossEntropyLoss`, which will
normalize the logits and compute the prediction error.

## Exercise 6 - Define the loss function

**Loss funtion**: Declare the variable `loss_fn` and assign it the value of `nn.CrossEntropyLoss()`

<details><summary><b>Solution</b></summary> <pre>

```python
loss_fn = nn.CrossEntropyLoss()
```
</pre></details>

Optimization Loop
=================

Once we set our hyperparameters, we can then train and optimize our
model with an optimization loop. Each iteration of the optimization loop
is called an **epoch**.

Each epoch consists of two main parts:

-   **The Train Loop** - iterate over the training dataset and try
        to converge to optimal parameters.
-   **The Validation/Test Loop** - iterate over the test dataset to
        check if model performance is improving.



Optimizer
=========

Optimization is the process of adjusting model parameters to reduce
model error in each training step. **Optimization algorithms** define
how this process is performed (in this example we use Stochastic
Gradient Descent). All optimization logic is encapsulated in the
`optimizer` object. Here, we use the SGD optimizer; additionally, there
are many [different
optimizers](https://pytorch.org/docs/stable/optim.html) available in
PyTorch such as ADAM and RMSProp, that work better for different kinds
of models and data.

We initialize the optimizer by registering the model\'s parameters that
need to be trained, and passing in the learning rate hyperparameter.

## Exercise 7 - Define the Optimizer

**Optimizer**: Declare the `optimizer` variable and assign it the value of `torch.optim.SGD` with the model parameters and learning rate as arguments.

<details><summary><b>Solution</b></summary> <pre>

```python
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
```

</pre></details>

Inside the training loop, optimization happens in three steps:

-   Call `optimizer.zero_grad()` to reset the gradients of model
        parameters. Gradients by default add up; to prevent
        double-counting, we explicitly zero them at each iteration.
-   Backpropagate the prediction loss with a call to
        `loss.backward()`. PyTorch deposits the gradients of the loss
        w.r.t. each parameter.
-   Once we have our gradients, we call `optimizer.step()` to adjust
        the parameters by the gradients collected in the backward pass.

Full Implementation
===================

We define `train` function that loops over our optimization code, and
`test` function that evaluates the model\'s performance against our test
data.

## Exercise 8 - Define the Training Function

1. **Define the Training Function**: Implement the `train` function that takes a dataloader, model, loss function, and optimizer as inputs.
2. **Set Model to Training Mode**: Ensure the model is set to training mode.
3. **Compute Predictions and Loss**: For each batch in the dataloader, compute the model's predictions and the corresponding loss.
4. **Perform Backpropagation**: Perform backpropagation to compute gradients and update the model parameters.
5. **Print Training Progress**: Print the loss and progress at regular intervals.



<details><summary><b>Solution</b></summary> <pre>

```python
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # Compute prediction and loss
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
```

</pre></details>

## Exercise 9 - Define the Testing Function

1. **Define the Testing Function**: Implement the `test` function that takes a dataloader, model, and loss function as inputs.
2. **Set Model to Evaluation Mode**: Ensure the model is set to evaluation mode.
3. **Compute Predictions and Loss**: For each batch in the dataloader, compute the model's predictions and the corresponding loss.
4. **Calculate Accuracy**: Calculate the accuracy of the model based on the predictions.
5. **Print Test Results**: Print the average loss and accuracy of the model.

<details><summary><b>Solution</b></summary> <pre>

```python
def test(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    # Unnecessary in this situation but added for best practices
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
```

</pre></details>

We initialize the loss function and optimizer, and pass it to
`train_loop` and `test_loop`. Feel free to increase the number of epochs
to track the model\'s improving performance.


In [ ]:
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Saving and Loading Model Weights
================================

PyTorch models store the learned parameters in an internal state
dictionary, called `state_dict`. These can be persisted via the
`torch.save` method:

```python
torch.save(model.state_dict(), "../models/model.pth")
print("Saved PyTorch Model State to ../models/model.pth")
```



Loading Models
==============
The process for loading a model includes re-creating the model structure and loading the state dictionary into it.

```python

model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("../models/model.pth"))
```